# 09. Apply Clock: Age Prediction on New Samples

A self-contained application pipeline. Given **new BAM, PAT, or beta files**, this notebook will:

1. Convert inputs to beta format (BAM → bam2pat → pat2beta, or PAT → pat2beta)
2. Optionally run LLR digital sorting to produce cell-type betas
3. Load the trained compact reference from GCS
4. Run maximum-likelihood age inference (K top CpGs by Fisher t-statistic)
5. Write a tidy CSV of per-sample predictions

**Only cells 1 and 2 need to be edited.** Everything else is automatic.

---

| Input format | Sorting needed? | Reference type |
|---|---|---|
| BAM | automatic | bulk or cell-type |
| PAT (bulk) | optional | bulk or cell-type |
| BETA (bulk) | — | bulk only |
| BETA (cell-sorted) | already done | cell-type |

---

## Step 1 — Configuration
Edit this cell to point at your inputs and choose which clock(s) to apply.

In [ ]:
import os
WORKSPACE_BUCKET = os.environ['WORKSPACE_BUCKET']

# ── INPUT ─────────────────────────────────────────────────────────────────
# Format: 'BAM', 'PAT', or 'BETA'
INPUT_FORMAT = 'BAM'

# Provide either a list of explicit GCS paths, or a GCS prefix to discover all files.
# For BAM: must end in .bam; for PAT: must end in .pat.gz; for BETA: must end in .beta
INPUT_PATHS  = []              # e.g. ['gs://bucket/bam/sample1.bam']
INPUT_PREFIX = None            # e.g. 'gs://bucket/my_new_cohort/bam/' (used if INPUT_PATHS is empty)

# ── CLOCK SETTINGS ────────────────────────────────────────────────────────
# Which clock(s) to apply. Options:
#   'bulk'                          — whole-blood bulk reference
#   'Myeloid', 'Lymphoid', 'T_Cell', 'B_Cell', 'NK_Cell', 'Monocyte', 'Granulocyte'
#   'all'                           — bulk + all 7 cell-type clocks
CLOCK_TYPES = ['bulk']

# Which reference group to use within each clock type.
# 'all_samples' uses the full training cohort (recommended).
# 'ONT', 'PacBio_Revio', 'PacBio_Sequel' uses the technology-matched reference.
REFERENCE_GROUP_LEVEL = 'technology'   # 'all_samples' | 'technology' | 'batch'
REFERENCE_GROUP_NAME  = 'ONT'          # e.g. 'all_samples', 'ONT', 'PacBio_Revio'

# GCS location of the reference index (built by notebook 05)
REFERENCE_GCS_ROOT = f'{WORKSPACE_BUCKET}/results/age_references_v2'

# Number of top CpG sites to use for inference (ranked by Fisher t-statistic)
K_INFER = 1000

# ── SORTING SETTINGS (only used when INPUT_FORMAT is BAM or PAT) ──────────
# Set to True to also produce cell-type-specific predictions via LLR sorting.
# Requires marker BEDs from notebook 03 to be present in GCS.
RUN_SORTING = True  # if False, only a bulk prediction is made

MARKERS_GCS_ROOT = f'{WORKSPACE_BUCKET}/results/markers/markers_S2'
# Coarse markers for Myeloid / Lymphoid split:
COARSE_MARKER_BEDS = {
    'Myeloid':  f'{MARKERS_GCS_ROOT}/coarse/Myeloid_vs_Lymphoid/Markers.Myeloid.bed',
    'Lymphoid': f'{MARKERS_GCS_ROOT}/coarse/Lymphoid_vs_Myeloid/Markers.Lymphoid.bed',
}
# Fine-grained one-vs-all markers:
HIER_MARKER_BEDS = {
    'T_Cell':     f'{MARKERS_GCS_ROOT}/hier/Lymphoid_T_vs_restLymph/Markers.T_Cell.bed',
    'B_Cell':     f'{MARKERS_GCS_ROOT}/hier/Lymphoid_B_vs_restLymph/Markers.B_Cell.bed',
    'NK_Cell':    f'{MARKERS_GCS_ROOT}/hier/Lymphoid_NK_vs_restLymph/Markers.NK_Cell.bed',
    'Monocyte':   f'{MARKERS_GCS_ROOT}/hier/Myeloid_Mono_vs_restMyeloid/Markers.Monocyte.bed',
    'Granulocyte':f'{MARKERS_GCS_ROOT}/hier/Myeloid_Gran_vs_restMyeloid/Markers.Granulocyte.bed',
}
ALL_TARGET_BEDS = {**COARSE_MARKER_BEDS, **HIER_MARKER_BEDS}

# Sorting thresholds (match notebook 04)
SORT_MIN_HITS = 5
SORT_TAU      = 1.053

# ── OUTPUT ────────────────────────────────────────────────────────────────
OUTPUT_GCS_PREFIX = f'{WORKSPACE_BUCKET}/results/clock_application'
OUTPUT_CSV_NAME   = 'age_predictions.csv'

print('Configuration loaded.')
print(f'  INPUT_FORMAT          : {INPUT_FORMAT}')
print(f'  CLOCK_TYPES           : {CLOCK_TYPES}')
print(f'  REFERENCE_GROUP_LEVEL : {REFERENCE_GROUP_LEVEL}')
print(f'  REFERENCE_GROUP_NAME  : {REFERENCE_GROUP_NAME}')
print(f'  RUN_SORTING           : {RUN_SORTING}')
print(f'  K_INFER               : {K_INFER}')


## Step 2 — Imports and environment check

In [ ]:
import io, re, math, gzip, shutil, subprocess, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm.auto import tqdm
from google.cloud import storage

client = storage.Client()

# wgbstools + bgzip — required only for BAM/PAT input
HOME     = Path(os.getcwd())
TOOL_BIN = HOME / 'wgbs_tools' / 'wgbstools'
CPG_CHROME_SIZE = HOME / 'wgbs_tools' / 'references' / 'hg38' / 'CpG.chrome.size'
CPG_BED_GZ      = HOME / 'wgbs_tools' / 'references' / 'hg38' / 'CpG.bed.gz'

TMP = HOME / 'tmp_apply_clock'
TMP.mkdir(exist_ok=True)

if INPUT_FORMAT in ('BAM', 'PAT'):
    assert TOOL_BIN.exists(),        f'wgbstools not found: {TOOL_BIN}'
    assert CPG_CHROME_SIZE.exists(), f'CpG.chrome.size not found: {CPG_CHROME_SIZE}'
    assert shutil.which('bgzip'),    'bgzip not found (apt-get install tabix)'
    print('wgbstools OK:', TOOL_BIN)
    print('bgzip      OK')

print('Imports OK.')


## Step 3 — GCS helpers

In [ ]:
def gs_to_bucket_blob(gs_path):
    assert gs_path.startswith('gs://'), gs_path
    rest = gs_path[5:]
    bkt, obj = rest.split('/', 1)
    return bkt, obj

def gcs_exists(gs_path):
    bkt, obj = gs_to_bucket_blob(gs_path)
    return client.bucket(bkt).blob(obj).exists()

def gcs_ls(prefix):
    'List all GCS objects under a prefix. Returns list of gs:// paths.'
    r = subprocess.run(['gsutil', 'ls', prefix], capture_output=True, text=True)
    return [l.strip() for l in r.stdout.splitlines() if l.strip()]

def download_text_from_gcs(gs_path):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    return blob.download_as_text() if blob.exists() else None

def load_npy_from_gcs(gs_path):
    bkt, obj = gs_to_bucket_blob(gs_path)
    data = client.bucket(bkt).blob(obj).download_as_bytes()
    return np.load(io.BytesIO(data), allow_pickle=False)

def load_json_from_gcs(gs_path):
    import json as _json
    txt = download_text_from_gcs(gs_path)
    if txt is None: raise FileNotFoundError(gs_path)
    return _json.loads(txt)

def load_beta_pairs_from_gcs(gs_path, dtype=np.uint8, min_bytes=1000):
    bkt, obj = gs_to_bucket_blob(gs_path)
    data = client.bucket(bkt).blob(obj).download_as_bytes()
    if not data or len(data) < min_bytes:
        raise ValueError(f'Empty/tiny beta: {gs_path}')
    arr = np.frombuffer(data, dtype=dtype)
    if arr.size % 2: arr = arr[:-1]
    return arr.reshape(-1, 2)

def upload_file_to_gcs(local_path, gs_path, content_type='application/octet-stream'):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    blob.chunk_size = 16 * 1024 * 1024
    with open(local_path, 'rb') as f:
        blob.upload_from_file(f, content_type=content_type, timeout=1800)

print('GCS helpers defined.')


## Step 4 — Load reference index and compact references

In [ ]:
import json as _json

REFERENCE_INDEX_GS = f'{REFERENCE_GCS_ROOT}/reference_index.csv'

def load_reference_index():
    txt = download_text_from_gcs(REFERENCE_INDEX_GS)
    if not txt:
        raise FileNotFoundError(
            f'Reference index not found: {REFERENCE_INDEX_GS}\n'
            'Run notebook 05 first to build the references.'
        )
    return pd.read_csv(io.StringIO(txt))

def load_compact_reference(compact_prefix,
                           age_grid_min=0.0, age_grid_max=100.0, age_step=0.1):
    'Load compact reference arrays from GCS and build inference age grid.'
    ref = {
        'eligible_idx':        load_npy_from_gcs(f'{compact_prefix}eligible_idx.npy').astype(np.int32),
        'intercept_eligible':  load_npy_from_gcs(f'{compact_prefix}intercept_eligible.npy').astype(np.float32),
        'coef_eligible':       load_npy_from_gcs(f'{compact_prefix}coef_eligible.npy').astype(np.float32),
        't_stat_eligible':     load_npy_from_gcs(f'{compact_prefix}t_stat_eligible.npy').astype(np.float32),
    }
    ref['age_grid'] = np.arange(
        age_grid_min, age_grid_max + age_step / 2, age_step, dtype=np.float32
    )
    scalars = load_json_from_gcs(f'{compact_prefix}scalars.json')
    ref['EPS'] = float(scalars['EPS'])
    ref['CAP'] = float(scalars['CAP'])
    return ref

def select_compact_reference(ref_index, modality, celltype,
                             group_level, group_name):
    'Find and load one compact reference from the index.'
    hit = ref_index[
        (ref_index['modality']    == modality)  &
        (ref_index['celltype']    == celltype)  &
        (ref_index['group_level'] == group_level) &
        (ref_index['group_name']  == group_name)
    ]
    if len(hit) == 0:
        raise ValueError(
            f'No reference found for {modality}/{celltype}/{group_level}/{group_name}.\n'
            f'Available combinations:\n{ref_index[["modality","celltype","group_level","group_name"]].to_string()}'
        )
    return load_compact_reference(hit.iloc[0]['compact_prefix'])

# ── Load index and resolve requested references ──────────────────────────
print('Loading reference index...')
ref_index = load_reference_index()
print(f'  {len(ref_index)} references available')
display(ref_index[['modality','celltype','group_level','group_name',
                    'n_train_samples','n_sites_eligible']].head(20))

# Expand 'all' clock types
ALL_CELL_TYPES = ['Myeloid','Lymphoid','T_Cell','B_Cell','NK_Cell','Monocyte','Granulocyte']
if CLOCK_TYPES == 'all' or 'all' in CLOCK_TYPES:
    _resolved = ['bulk'] + ALL_CELL_TYPES
else:
    _resolved = list(CLOCK_TYPES)

# Pre-load all requested compact references
loaded_refs = {}
for ct in _resolved:
    modality = 'bulk' if ct == 'bulk' else 'celltype'
    celltype = 'bulk' if ct == 'bulk' else ct
    try:
        loaded_refs[ct] = select_compact_reference(
            ref_index, modality, celltype,
            REFERENCE_GROUP_LEVEL, REFERENCE_GROUP_NAME
        )
        n_sites = int(loaded_refs[ct]['eligible_idx'].shape[0])
        print(f'  Loaded {ct} reference: {n_sites:,} eligible sites')
    except ValueError as e:
        print(f'  WARNING: {e}')

print(f'\nLoaded {len(loaded_refs)} reference(s): {list(loaded_refs.keys())}')


## Step 5 — Inference engine

In [ ]:
def predict_from_beta_path(beta_gcs_path, compact_ref, K=1000):
    'Run ML age inference on one beta file. Returns (pred_age, n_sites_used, status).'
    try:
        pairs = load_beta_pairs_from_gcs(beta_gcs_path)
    except Exception as e:
        return float('nan'), 0, f'load_error:{e}'

    eligible_idx = compact_ref['eligible_idx']
    if eligible_idx.size == 0:
        return float('nan'), 0, 'no_eligible_sites'

    m = pairs[:, 0].astype(np.float32)
    t = pairs[:, 1].astype(np.float32)

    available = t[eligible_idx] > 0
    if available.sum() == 0:
        return float('nan'), 0, 'no_overlap'

    idx_local  = np.where(available)[0]
    order      = np.argsort(compact_ref['t_stat_eligible'][idx_local])[::-1]
    idx_local  = idx_local[order][:min(K, len(idx_local))]

    if len(idx_local) < max(1, min(10, K)):
        return float('nan'), int(len(idx_local)), 'too_few_sites'

    chosen = eligible_idx[idx_local]
    m_raw  = m[chosen]
    t_raw  = t[chosen]
    t_use  = np.minimum(t_raw, compact_ref['CAP'])
    scale  = np.divide(t_use, t_raw,
                       out=np.ones_like(t_raw), where=t_raw > 0)
    m_use  = m_raw * scale

    b0  = compact_ref['intercept_eligible'][idx_local]
    b1  = compact_ref['coef_eligible'][idx_local]
    eps = compact_ref['EPS']
    age_grid = compact_ref['age_grid']

    ll = np.zeros(len(age_grid), dtype=np.float32)
    u_use = t_use - m_use
    for i, a in enumerate(age_grid):
        p  = np.clip(b0 + b1 * float(a), eps, 1.0 - eps)
        ll[i] = np.sum(m_use * np.log(p) + u_use * np.log(1.0 - p))

    best_age = float(age_grid[np.argmax(ll)])
    return best_age, int(len(idx_local)), 'ok'

print('Inference engine defined.')
print(f'  Using K={K_INFER} top CpGs per sample per clock.')


## Step 6 — Discover input files

In [ ]:
def discover_inputs(format_: str, explicit_paths, prefix):
    'Return list of GCS paths matching the requested input format.'
    ext = {'BAM': '.bam', 'PAT': '.pat.gz', 'BETA': '.beta'}[format_]

    if explicit_paths:
        paths = [p for p in explicit_paths if p.endswith(ext)]
        if len(paths) < len(explicit_paths):
            print(f'WARNING: {len(explicit_paths) - len(paths)} paths ignored (wrong extension)')
        return paths

    if prefix:
        all_objects = gcs_ls(prefix.rstrip('/') + '/')
        return [p for p in all_objects if p.endswith(ext)]

    raise ValueError('Provide either INPUT_PATHS or INPUT_PREFIX.')

input_files = discover_inputs(INPUT_FORMAT, INPUT_PATHS, INPUT_PREFIX)
print(f'Found {len(input_files)} {INPUT_FORMAT} file(s):')
for p in input_files[:10]:
    print(f'  {p}')
if len(input_files) > 10:
    print(f'  ... and {len(input_files) - 10} more')

if not input_files:
    raise ValueError(
        f'No {INPUT_FORMAT} files found. '
        'Check INPUT_PATHS / INPUT_PREFIX and INPUT_FORMAT.'
    )


## Step 7 — BAM → PAT conversion
_Skipped automatically when `INPUT_FORMAT` is `'PAT'` or `'BETA'`._

In [ ]:
def bam_to_pat(bam_gcs_path, out_prefix_gcs, threads=4, skip_if_exists=True):
    'Convert one BAM to PAT via wgbstools bam2pat. Returns GCS path of .pat.gz.'
    sample_id = Path(bam_gcs_path).stem  # removes .bam
    pat_gcs   = f'{out_prefix_gcs}/{sample_id}.pat.gz'

    if skip_if_exists and gcs_exists(pat_gcs):
        print(f'  [skip] {sample_id} — PAT already exists')
        return pat_gcs

    local_bam = TMP / f'{sample_id}.bam'
    local_pat = TMP / f'{sample_id}.pat'

    try:
        print(f'  Downloading {sample_id}.bam ...')
        subprocess.run(['gsutil', '-q', 'cp', bam_gcs_path, str(local_bam)], check=True)
        subprocess.run(['gsutil', '-q', 'cp', bam_gcs_path + '.bai', str(local_bam) + '.bai'],
                       check=False)  # index optional — wgbstools may regenerate

        print(f'  Running bam2pat ...')
        subprocess.run([
            str(TOOL_BIN), 'bam2pat',
            '--bam',         str(local_bam),
            '--out_dir',     str(TMP),
            '--threads',     str(threads),
            '--genome',      'hg38',
        ], check=True)

        pat_gz = TMP / f'{sample_id}.pat.gz'
        assert pat_gz.exists(), f'bam2pat did not produce {pat_gz}'

        print(f'  Uploading {sample_id}.pat.gz ...')
        upload_file_to_gcs(str(pat_gz), pat_gcs)
        return pat_gcs
    finally:
        for f in [local_bam, local_bam.with_suffix('.bai'),
                  TMP / f'{sample_id}.pat', TMP / f'{sample_id}.pat.gz',
                  TMP / f'{sample_id}.pat.gz.csi']:
            if Path(f).exists(): Path(f).unlink()

# Convert all BAMs → PATs
if INPUT_FORMAT == 'BAM':
    print(f'Converting {len(input_files)} BAM file(s) to PAT ...')
    pat_out_prefix = f'{OUTPUT_GCS_PREFIX}/pat'
    pat_files = []
    for bam_path in tqdm(input_files):
        pat_path = bam_to_pat(bam_path, pat_out_prefix)
        pat_files.append(pat_path)
    print(f'PAT conversion done: {len(pat_files)} files')
elif INPUT_FORMAT == 'PAT':
    pat_files = list(input_files)
    print(f'Using {len(pat_files)} pre-existing PAT files')
else:
    pat_files = []  # BETA input — no PAT step
    print('BETA input: skipping BAM→PAT step')


## Step 8 — LLR digital sorting
_Skipped when `RUN_SORTING = False` or `INPUT_FORMAT = 'BETA'`._

Assigns each read to its most likely cell type using the LLR scorer from notebook 04. Produces one `.pat.gz` + `.beta` per (sample, cell type).

In [ ]:
assert TOOL_BIN.exists(),        f"wgbstools not found: {TOOL_BIN}"
assert CPG_CHROME_SIZE.exists(), f"CpG.chrome.size not found: {CPG_CHROME_SIZE}"
assert shutil.which("bgzip"),    "bgzip not found — install: sudo apt-get install tabix"
print("wgbstools OK:", TOOL_BIN)
print("bgzip      OK")


def load_cpg_chrom_sizes(path: Path):
    'Parse CpG.chrome.size -> (cs DataFrame, chrom_counts dict, chrom_offsets dict).'
    cs = pd.read_csv(path, sep="\t", header=None, names=["chrom", "n_cpg"])
    cs["n_cpg"]   = cs["n_cpg"].astype(np.int64)
    cs["offset"]  = cs["n_cpg"].cumsum().shift(fill_value=0).astype(np.int64)
    chrom_counts  = dict(zip(cs["chrom"], cs["n_cpg"]))
    chrom_offsets = dict(zip(cs["chrom"], cs["offset"]))
    return cs, chrom_counts, chrom_offsets


cs, chrom_counts, chrom_offsets = load_cpg_chrom_sizes(CPG_CHROME_SIZE)
print(f"Total CpGs in hg38: {int(cs['n_cpg'].sum()):,}")
display(cs.head(5))

In [ ]:
def marker_interval_to_local_slice(chrom, cpg_start_1based, cpg_end_1based,
                                   chrom_counts, chrom_offsets):
    'Convert a marker BED interval to (j0, j1) 0-based slice; None if invalid.'
    if chrom not in chrom_counts:
        return None
    n   = int(chrom_counts[chrom])
    off = int(chrom_offsets[chrom])
    if 1 <= cpg_start_1based <= n and 1 <= cpg_end_1based <= n + 1:
        local_start, local_end = cpg_start_1based, cpg_end_1based
    elif off + 1 <= cpg_start_1based <= off + n and off + 1 <= cpg_end_1based <= off + n + 1:
        local_start = cpg_start_1based - off
        local_end   = cpg_end_1based   - off
    else:
        return None
    j0 = max(0, min(int(local_start - 1), n))
    j1 = max(0, min(int(local_end   - 1), n))
    if j1 <= j0:
        return None
    return j0, j1


def pat_start_to_local_j(chrom, start_cpg_1based, chrom_counts, chrom_offsets, base=1):
    'Convert PAT start CpG index (1-based, local or global) to 0-based local j.'
    if chrom not in chrom_counts:
        return None
    n   = int(chrom_counts[chrom])
    off = int(chrom_offsets[chrom])
    if 1 <= start_cpg_1based <= n:
        local_1based = start_cpg_1based
    elif off + 1 <= start_cpg_1based <= off + n:
        local_1based = start_cpg_1based - off
    else:
        return None
    j = int(local_1based - base)
    if j < 0 or j >= n:
        return None
    return j


print("Index mapping helpers defined.")

In [ ]:
def build_chrom_params_target_bg(markers_bed, chrom_counts, chrom_offsets, eps=1e-4):
    '
    Parse marker BED and build per-chromosome probability arrays.

    BED columns: 0=chrom, 3=cpg_start, 4=cpg_end, 9=p_tg, 10=p_bg.
    Returns (chrom_params dict, n_marker_sites int, df_shape tuple).
    '
    df = pd.read_csv(markers_bed, sep="\t", comment="#", header=None)
    if df.shape[1] < 11:
        raise ValueError(f"Marker BED has only {df.shape[1]} columns; need >= 11")

    use = df[[0, 3, 4, 9, 10]].copy()
    use.columns = ["chrom", "cpg_start", "cpg_end", "p_tg", "p_bg"]
    use["cpg_start"] = use["cpg_start"].astype(np.int64)
    use["cpg_end"]   = use["cpg_end"].astype(np.int64)
    use["p_tg"]      = use["p_tg"].astype(np.float32)
    use["p_bg"]      = use["p_bg"].astype(np.float32)

    chrom_params = {
        chrom: {
            "p_tg": np.full(int(n), np.nan, dtype=np.float32),
            "p_bg": np.full(int(n), np.nan, dtype=np.float32),
        }
        for chrom, n in chrom_counts.items()
    }
    for chrom, g in use.groupby("chrom"):
        if chrom not in chrom_params:
            continue
        p_tg_arr = chrom_params[chrom]["p_tg"]
        p_bg_arr = chrom_params[chrom]["p_bg"]
        for cpg_s, cpg_e, ptg, pbg in zip(g["cpg_start"].values, g["cpg_end"].values,
                                           g["p_tg"].values, g["p_bg"].values):
            sl = marker_interval_to_local_slice(
                chrom, int(cpg_s), int(cpg_e), chrom_counts, chrom_offsets)
            if sl is None:
                continue
            j0, j1 = sl
            p_tg_arr[j0:j1] = float(np.clip(ptg, eps, 1.0 - eps))
            p_bg_arr[j0:j1] = float(np.clip(pbg, eps, 1.0 - eps))

    total_sites = sum(int(np.isfinite(chrom_params[c]["p_tg"]).sum()) for c in chrom_params)
    return chrom_params, total_sites, df.shape


print("build_chrom_params_target_bg defined.")

In [ ]:
PAT_LINE_RE  = re.compile(r"^(\S+)\t(\d+)\t(\S+)\t(\d+)\s*$")
METH_CHARS   = frozenset(["C", "1"])
UNMETH_CHARS = frozenset(["T", "0"])


def parse_pat_line(line: str):
    "Parse one PAT line. Returns (chrom, start_cpg, pattern, multiplicity) or None."
    m = PAT_LINE_RE.match(line)
    if not m:
        return None
    return m.group(1), int(m.group(2)), m.group(3), int(m.group(4))


def infer_pat_base(pat_gz: Path, n_lines: int = 20_000) -> int:
    "Inspect CpG start indices to detect local vs global coordinates (both 1-based)."
    mn = math.inf
    with gzip.open(pat_gz, "rt") as f:
        for i, line in enumerate(f):
            if i >= n_lines:
                break
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            try:
                mn = min(mn, int(parts[1]))
            except ValueError:
                pass
    return 1 if mn >= 1 else 0


print("PAT parsing helpers defined.")

In [ ]:
def llr_for_row_sparse(pattern, p_tg_seg, p_bg_seg, eps=1e-4):
    '
    Compute LLR for one PAT read against a target/background marker segment.
    Returns (llr float, hits int).
    '
    mask = np.isfinite(p_tg_seg) & np.isfinite(p_bg_seg)
    if not mask.any():
        return 0.0, 0
    idxs  = np.flatnonzero(mask)
    pt    = np.clip(p_tg_seg[idxs], eps, 1.0 - eps)
    pb    = np.clip(p_bg_seg[idxs], eps, 1.0 - eps)
    log_m = np.log(pt)    - np.log(pb)
    log_u = np.log1p(-pt) - np.log1p(-pb)
    llr  = 0.0
    hits = 0
    for k, i in enumerate(idxs):
        ch = pattern[i]
        if ch in METH_CHARS:
            llr  += float(log_m[k]);  hits += 1
        elif ch in UNMETH_CHARS:
            llr  += float(log_u[k]);  hits += 1
    return llr, hits


print("LLR scoring function defined.")

In [ ]:
def bgzip_file(in_txt: Path) -> Path:
    "bgzip a plain-text PAT file in place. Returns path to the .gz output."
    out_gz = Path(str(in_txt) + ".gz")
    if out_gz.exists(): out_gz.unlink()
    subprocess.run(["bgzip", "-f", str(in_txt)], check=True)
    assert out_gz.exists(), f"bgzip failed: {out_gz}"
    return out_gz


def wgbstools_index(pat_gz: Path) -> Path:
    "Create a .csi tabix index for a bgzipped PAT file."
    csi = Path(str(pat_gz) + ".csi")
    if csi.exists(): csi.unlink()
    subprocess.run([str(TOOL_BIN), "index", str(pat_gz)], check=True)
    assert csi.exists(), f"wgbstools index failed: {csi}"
    return csi


def pat2beta(pat_gz, out_dir, genome="hg38", threads=4, force=True):
    "Run wgbstools pat2beta. Returns path to .beta output."
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(TOOL_BIN), "pat2beta"]
    if force:
        cmd.append("-f")
    cmd += ["--genome", genome, "--threads", str(threads),
            "--out_dir", str(out_dir), str(pat_gz)]
    subprocess.run(cmd, check=True)
    stem = pat_gz.name
    if stem.endswith(".pat.gz"): stem = stem[:-7]
    cand = out_dir / f"{stem}.beta"
    if cand.exists(): return cand
    betas = sorted(out_dir.glob("*.beta"), key=lambda p: p.stat().st_mtime, reverse=True)
    if betas: return betas[0]
    raise FileNotFoundError(f"pat2beta produced no .beta in {out_dir}")


print("bgzip / index / pat2beta helpers defined.")

In [ ]:
def process_one_sample(pat_gcs, out_root_gcs, min_hits=5, tau=1.053,
                       threads=4, make_beta=True, skip_if_exists=True):
    'Sort one bulk PAT file into 7 cell-type-specific PAT (+ beta) files.'
    sample_id  = Path(pat_gcs).name.replace(".pat.gz", "")
    sample_key = make_sample_key(pat_gcs)
    tau_log    = math.log(tau)
    print(f"\n=== {sample_key} ===")

    if skip_if_exists:
        probe = f"{out_root_gcs}/Myeloid/{sample_key}_Myeloid.pat.gz"
        if gcs_exists(probe):
            print(f"  Skip (exists): {probe}")
            return {"sample_id": sample_id, "sample_key": sample_key, "status": "skipped_exists"}

    local_in = TMP / f"{sample_key}.pat.gz"
    print(f"  Downloading {pat_gcs} ...")
    subprocess.run(["gsutil", "-m", "-q", "cp", pat_gcs, str(local_in)], check=True)
    base = infer_pat_base(local_in)

    out_txt = {}
    out_fh  = {}
    for target in TARGET_BEDS:
        p = TMP / f"{sample_key}_{safe_label(target)}.pat"
        if p.exists(): p.unlink()
        out_txt[target] = p
        out_fh[target]  = open(p, "wt")

    counts    = {t: {"Target": 0, "Other": 0, "AmbiguousTie": 0,
                     "LowHits": 0, "NoOverlap": 0, "Skipped": 0}
                 for t in TARGET_BEDS}
    hits_hist = {t: Counter() for t in TARGET_BEDS}

    with gzip.open(local_in, "rt") as fin:
        for line in fin:
            parsed = parse_pat_line(line)
            if parsed is None:
                for t in TARGET_BEDS: counts[t]["Skipped"] += 1
                continue
            chrom, start_cpg, pattern, n_reads = parsed
            if chrom not in chrom_counts:
                for t in TARGET_BEDS: counts[t]["Skipped"] += n_reads
                continue
            j = pat_start_to_local_j(chrom, start_cpg, chrom_counts, chrom_offsets, base=base)
            if j is None:
                for t in TARGET_BEDS: counts[t]["Skipped"] += n_reads
                continue
            L = len(pattern)
            if j + L > int(chrom_counts[chrom]):
                for t in TARGET_BEDS: counts[t]["Skipped"] += n_reads
                continue
            for target, params in chrom_params_by_target.items():
                p_tg_seg = params[chrom]["p_tg"][j : j + L]
                p_bg_seg = params[chrom]["p_bg"][j : j + L]
                llr, hits = llr_for_row_sparse(pattern, p_tg_seg, p_bg_seg)
                hits_hist[target][hits] += 1
                if hits == 0:
                    counts[target]["NoOverlap"] += n_reads
                elif hits < min_hits:
                    counts[target]["LowHits"] += n_reads
                elif llr > tau_log:
                    counts[target]["Target"] += n_reads
                    out_fh[target].write(line)
                elif llr < -tau_log:
                    counts[target]["Other"] += n_reads
                else:
                    counts[target]["AmbiguousTie"] += n_reads

    for fh in out_fh.values():
        fh.close()

    produced_targets = []
    for target in TARGET_BEDS:
        txt = out_txt[target]
        if not txt.exists() or txt.stat().st_size == 0:
            if txt.exists(): txt.unlink()
            continue
        gz  = bgzip_file(txt)
        csi = wgbstools_index(gz)
        produced = [gz, csi]
        if make_beta:
            beta_dir = TMP / f"{sample_key}_{safe_label(target)}_beta"
            beta     = pat2beta(gz, beta_dir, genome="hg38", threads=threads, force=True)
            produced.append(beta)
        dst = f"{out_root_gcs}/{safe_label(target)}/"
        subprocess.run(["gsutil", "-m", "-q", "cp", *[str(p) for p in produced], dst], check=True)
        print(f"  {target}: {counts[target]['Target']:>8,} target reads -> {dst}")
        cleanup_files([gz, csi])
        if make_beta: cleanup_files([beta_dir])
        produced_targets.append(target)

    cleanup_files([local_in])

    row = {"sample_id": sample_id, "sample_key": sample_key, "status": "ok",
           "min_hits": min_hits, "tau": tau, "made_targets": ",".join(produced_targets)}
    for target in TARGET_BEDS:
        h = hits_hist[target]
        row[f"{target}_median_hits"] = quantile_from_hist(h, 0.5)
        row[f"{target}_p90_hits"]    = quantile_from_hist(h, 0.9)
        for k, v in counts[target].items():
            row[f"{target}_{k}"] = int(v)
    return row


print("process_one_sample defined.")

In [ ]:
# ── Download marker BEDs and build probability atlases ─────────────────
if RUN_SORTING and pat_files:
    MARKERS_LOCAL = TMP / 'markers'
    MARKERS_LOCAL.mkdir(exist_ok=True)

    chrom_params_by_target = {}
    for target, gcs_bed in ALL_TARGET_BEDS.items():
        local_bed = MARKERS_LOCAL / f'{target}.bed'
        if not local_bed.exists():
            print(f'Downloading {target} marker BED...')
            r = subprocess.run(
                ['gsutil', '-q', 'cp', gcs_bed, str(local_bed)],
                capture_output=True
            )
            if r.returncode != 0:
                print(f'  WARNING: could not download {gcs_bed} — skipping {target}')
                continue
        params, n_sites, _ = build_chrom_params_target_bg(
            local_bed, chrom_counts, chrom_offsets
        )
        chrom_params_by_target[target] = params
        print(f'  {target}: {n_sites} marker sites')

    print(f'\nSorting {len(pat_files)} PAT file(s) into cell types...')
    sort_out_prefix = f'{OUTPUT_GCS_PREFIX}/cell_sorted'
    sort_stats = []
    for pat_gcs in tqdm(pat_files):
        try:
            stats = process_one_sample(
                pat_gcs=pat_gcs,
                out_root_gcs=sort_out_prefix,
                min_hits=SORT_MIN_HITS,
                tau=SORT_TAU,
                threads=4,
                make_beta=True,
                skip_if_exists=True,
            )
            sort_stats.append(stats)
        except Exception as e:
            print(f'  ERROR sorting {pat_gcs}: {e}')

    # Collect cell-type beta paths produced by sorting
    sorted_betas = {}  # {cell_type: [gcs_path, ...]}
    for target in ALL_TARGET_BEDS:
        paths = gcs_ls(f'{sort_out_prefix}/{target}/')
        sorted_betas[target] = [p for p in paths if p.endswith('.beta')]
        print(f'  {target}: {len(sorted_betas[target])} beta files')
else:
    sorted_betas = {}
    print('Sorting skipped.')


## Step 9 — PAT → bulk beta (pat2beta)
_Only for BAM/PAT input, producing the bulk beta for bulk-clock inference._

In [ ]:
def pat_to_bulk_beta(pat_gcs, out_prefix_gcs, skip_if_exists=True):
    'Convert a PAT file to a genome-wide bulk beta file. Returns GCS path.'
    sample_id = Path(pat_gcs).name.replace('.pat.gz', '')
    beta_gcs  = f'{out_prefix_gcs}/{sample_id}.beta'

    if skip_if_exists and gcs_exists(beta_gcs):
        return beta_gcs

    local_pat  = TMP / f'{sample_id}.pat.gz'
    local_beta = TMP / f'{sample_id}.beta'

    try:
        subprocess.run(['gsutil', '-q', 'cp', pat_gcs, str(local_pat)], check=True)
        subprocess.run([
            str(TOOL_BIN), 'pat2beta',
            str(local_pat),
            '--genome', 'hg38',
            '-o', str(local_beta),
        ], check=True)
        upload_file_to_gcs(str(local_beta), beta_gcs)
        return beta_gcs
    finally:
        for f in [local_pat, local_beta]:
            if Path(f).exists(): Path(f).unlink()

if INPUT_FORMAT in ('BAM', 'PAT') and 'bulk' in loaded_refs:
    print(f'Converting {len(pat_files)} PAT(s) to bulk beta...')
    bulk_out = f'{OUTPUT_GCS_PREFIX}/bulk_beta'
    bulk_betas = []
    for pat_gcs in tqdm(pat_files):
        b = pat_to_bulk_beta(pat_gcs, bulk_out)
        bulk_betas.append(b)
    print(f'  {len(bulk_betas)} bulk betas ready')
elif INPUT_FORMAT == 'BETA':
    bulk_betas = list(input_files)
    print(f'Using {len(bulk_betas)} pre-existing BETA files for bulk inference')
else:
    bulk_betas = []
    print('No bulk betas (bulk clock not requested or no PAT input)')


## Step 10 — Run inference

In [ ]:
def extract_sample_id(gcs_path, clock_type='bulk'):
    'Extract a clean sample ID from a GCS beta path.'
    name = Path(gcs_path).name
    for suffix in [f'_{clock_type}.beta', '.beta']:
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return name

all_rows = []

# ── Bulk inference ─────────────────────────────────────────────────────
if 'bulk' in loaded_refs and bulk_betas:
    print(f'Running bulk inference on {len(bulk_betas)} samples...')
    for beta_path in tqdm(bulk_betas, desc='bulk'):
        sample_id = extract_sample_id(beta_path)
        pred, n_sites, status = predict_from_beta_path(
            beta_path, loaded_refs['bulk'], K=K_INFER
        )
        all_rows.append({
            'sample_id':    sample_id,
            'clock_type':   'bulk',
            'pred_age':     pred,
            'n_sites_used': n_sites,
            'status':       status,
            'beta_path':    beta_path,
        })

# ── Cell-type inference ─────────────────────────────────────────────────
for ct, ref in loaded_refs.items():
    if ct == 'bulk':
        continue

    ct_betas = sorted_betas.get(ct, [])
    if not ct_betas:
        print(f'  No betas found for {ct} — skipping')
        continue

    print(f'Running {ct} inference on {len(ct_betas)} samples...')
    for beta_path in tqdm(ct_betas, desc=ct):
        sample_id = extract_sample_id(beta_path, ct)
        pred, n_sites, status = predict_from_beta_path(
            beta_path, ref, K=K_INFER
        )
        all_rows.append({
            'sample_id':    sample_id,
            'clock_type':   ct,
            'pred_age':     pred,
            'n_sites_used': n_sites,
            'status':       status,
            'beta_path':    beta_path,
        })

results_df = pd.DataFrame(all_rows)
ok_df = results_df[results_df['status'] == 'ok']

print(f'\nInference complete.')
print(f'  Total predictions : {len(results_df)}')
print(f'  Successful (ok)   : {len(ok_df)}')
print(f'  Failed            : {len(results_df) - len(ok_df)}')

display(results_df.head(20))


## Step 11 — Pivot to wide format
One row per sample; one column per clock type.

In [ ]:
ok_results = results_df[results_df['status'] == 'ok'].copy()

if not ok_results.empty:
    wide_df = ok_results.pivot_table(
        index='sample_id',
        columns='clock_type',
        values='pred_age',
        aggfunc='first'
    ).reset_index()
    wide_df.columns.name = None

    # Also add n_sites_used per clock
    for ct in ok_results['clock_type'].unique():
        sub = ok_results[ok_results['clock_type'] == ct].set_index('sample_id')['n_sites_used']
        wide_df[f'n_sites_{ct}'] = wide_df['sample_id'].map(sub)

    print('Wide results table:')
    display(wide_df)
else:
    wide_df = pd.DataFrame()
    print('No successful predictions to pivot.')


## Step 12 — Save results to GCS

In [ ]:
import json as _json

local_long  = TMP / 'age_predictions_long.csv'
local_wide  = TMP / 'age_predictions_wide.csv'
local_meta  = TMP / 'run_metadata.json'

gcs_long  = f'{OUTPUT_GCS_PREFIX}/{OUTPUT_CSV_NAME}'
gcs_wide  = f'{OUTPUT_GCS_PREFIX}/age_predictions_wide.csv'
gcs_meta  = f'{OUTPUT_GCS_PREFIX}/run_metadata.json'

results_df.to_csv(local_long, index=False)
if not wide_df.empty:
    wide_df.to_csv(local_wide, index=False)

meta = {
    'input_format':         INPUT_FORMAT,
    'clock_types':          list(loaded_refs.keys()),
    'reference_group_level': REFERENCE_GROUP_LEVEL,
    'reference_group_name':  REFERENCE_GROUP_NAME,
    'K_infer':              K_INFER,
    'run_sorting':          RUN_SORTING,
    'n_input_files':        len(input_files),
    'n_predictions_total':  len(results_df),
    'n_predictions_ok':     int((results_df['status'] == 'ok').sum()),
}
with open(local_meta, 'w') as f:
    _json.dump(meta, f, indent=2)

for local, gcs in [
    (local_long, gcs_long),
    (local_wide, gcs_wide),
    (local_meta, gcs_meta),
]:
    if Path(local).exists():
        subprocess.run(['gsutil', '-q', 'cp', str(local), gcs], check=True)
        print(f'  Saved: {gcs}')

print(f'\nDone. Results at: {OUTPUT_GCS_PREFIX}/')


## Step 13 — Summary visualization

In [ ]:
if ok_df.empty:
    print('No successful predictions — skipping plot.')
else:
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'pdf.fonttype': 42, 'ps.fonttype': 42,
        'figure.dpi': 150, 'savefig.dpi': 300,
    })

    clock_types_ok = sorted(ok_df['clock_type'].unique())
    n_clocks = len(clock_types_ok)
    ncols = min(3, n_clocks)
    nrows = (n_clocks + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5 * ncols, 4 * nrows),
                             facecolor='white', squeeze=False)

    for idx, ct in enumerate(clock_types_ok):
        ax = axes[idx // ncols][idx % ncols]
        sub = ok_df[ok_df['clock_type'] == ct]['pred_age']
        ax.hist(sub, bins=20, color='#185FA5', edgecolor='white', linewidth=0.5)
        ax.set_title(f'{ct}  (n={len(sub)})', fontsize=10, fontweight='bold')
        ax.set_xlabel('Predicted age (years)', fontsize=9)
        ax.set_ylabel('Samples', fontsize=9)
        ax.axvline(float(sub.median()), color='#C44E52', linestyle='--',
                   linewidth=1.2, label=f'Median: {sub.median():.1f} yr')
        ax.legend(fontsize=8, frameon=False)

    # Hide empty subplots
    for idx in range(n_clocks, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(
        f'Predicted ages — {len(ok_df["sample_id"].unique())} samples, '
        f'{n_clocks} clock(s)',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()

    local_fig = str(TMP / 'predicted_age_distribution.png')
    fig.savefig(local_fig, bbox_inches='tight', facecolor='white')
    subprocess.run(['gsutil', '-q', 'cp', local_fig,
                    f'{OUTPUT_GCS_PREFIX}/predicted_age_distribution.png'], check=False)
    plt.show()
    print(f'Plot saved to {OUTPUT_GCS_PREFIX}/predicted_age_distribution.png')
